In [ ]:
import os
import copy
import subprocess
from glob import glob
from itertools import product
from datetime import datetime
from pathlib import Path
from string import Template
from utils.notebook import isnotebook
if isnotebook():
    home_dir = os.path.expanduser("~")
    os.chdir(os.path.join(home_dir, "aiwq"))

    # Autoreload modified packages
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")

# Inline plotting setup
%matplotlib inline
%config InlineBackend.figure_formats = ['pdf', 'svg']
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
import cartopy.crs as ccrs
from matplotlib.colors import LinearSegmentedColormap
from IPython.display import Markdown, display
from AI_WQ_package import forecast_submission
from src.utils.data_io import *
from src.viz.viz_utils_pbc import *
from src.viz.viz_utils_extreme import generate_flood_bss_barplot
from models.utils.general_util import printf
from models.utils.eval_util import get_target_dates
from models.utils.data_utils import get_measurement_variable
from models.utils.models_util import get_submodel_name, get_selected_submodel_name
from utils.timing import tic, toc
from utils.data_io import save_to_netcdf, load_data
from utils.logging import printf


## Fig. 2: Global forecast skill of the leading dynamical model (ECMWF) and its probabilistic bias correction (PBC). 

#### Fig. 2a: Raw ECMWF, Debiased ECMWF, PBC-ECMWF barplots by task (2016-2024)

In [ ]:
fig_model_names=['climatology', 'ecmwf', 'debiased_ecmwf', 'pbc_ecmwf_combo']
fig_model_names_str="ECMWF models"
fig_gt_ids=["era5-tas", "era5-pr", "era5-mslp"]
fig_horizons=["19", "26"]
fig_target_dates="std_test"
fig_target_dates_list=[fig_target_dates]
fig_verbose=False


metric_dic = get_all_rps(model_names=fig_model_names,
                      model_names_str=fig_model_names_str,
                      gt_ids=fig_gt_ids,
                      horizons=fig_horizons,
                      target_dates_list=fig_target_dates_list,
                      verbose=fig_verbose)
metric_dic = {task: metric_dic[(task,target_dates)] for task, target_dates in metric_dic.keys()}

if False:
    print_improvements(metric_dic, 
                       model_name='pbc_ecmwf_combo', 
                       baseline_models=['ecmwf', 'debiased_ecmwf'])

fig_show=True
fig_save=True
fig_by_season=False
fig_verbose=False
figure_source_data = True
figure_source_data_filename = "fig2_pbc_ecmwf.xlsx"
figure_source_data_sheet_prefix="fig2a"
plot_rpss_barplot(metric_dic,
                   model_names=fig_model_names,
                   target_dates=fig_target_dates,
                   show_fig=fig_show,
                   save_fig=fig_save,
                   by_season=fig_by_season,
                   source_data=figure_source_data,
                   source_data_filename=figure_source_data_filename,
                   source_data_sheet_prefix=figure_source_data_sheet_prefix,
                   verbose=fig_verbose)

#### Fig. 2b: Raw ECMWF, Debiased ECMWF, PBC-ECMWF Diff maps (2016-2024)

In [ ]:
fig_model_names=['climatology', 'ecmwf', 'debiased_ecmwf', 'pbc_ecmwf_combo']
fig_model_names_str="ECMWF models"
fig_gt_ids=["era5-tas", "era5-pr", "era5-mslp"]
fig_horizons=["19", "26"]
fig_target_dates=["std_test"]
fig_verbose=False


# Set figure parameters
figure_model_names = ['ecmwf', 'debiased_ecmwf', 'pbc_ecmwf_combo']
figure_gt_ids = ['era5-tas', 'era5-pr', 'era5-mslp']
figure_horizons = [19, 26]
figure_metric = 'lat_lon_rpss'
figure_target_dates = 'std_test'
diff_cmap = "bwr"
skill_cmap = "RdBu_r"
figure_show = True
figure_save = True
figure_source_data = True
figure_source_data_filename = "fig2_pbc_ecmwf.xlsx"
figure_source_data_sheet_prefix="fig2b"

plot_metric_diff_grid_6x4(model_names=figure_model_names,
                        gt_ids=figure_gt_ids,
                        horizons=figure_horizons,
                        metric=figure_metric,
                        target_dates=figure_target_dates,
                        diff_cmap=diff_cmap,
                        skill_cmap=skill_cmap,   
                        show_fig=figure_show,
                        source_data=figure_source_data,
                        source_data_filename=figure_source_data_filename,
                        source_data_sheet_prefix=figure_source_data_sheet_prefix,
                        save_fig=figure_save)

## Fig. 3: Global forecast skill of AI models and their probabilistic bias corrections (PBC).

#### Fig. 3a: AIFS, PBC-AIFS barplots by task (2025)

In [ ]:
fig_model_names=['climatology', 'debiased_aifs', 'pbc_debias_aifs']
fig_model_names_str="AIFS models"
fig_gt_ids=["era5-tas", "era5-pr", "era5-mslp"]
fig_horizons=["19", "26"]
fig_target_dates="std_aifs_forecast"
fig_target_date_list=[fig_target_dates]
fig_verbose=False

metric_dic = get_all_rps(model_names=fig_model_names,
                      model_names_str=fig_model_names_str,
                      gt_ids=fig_gt_ids,
                      horizons=fig_horizons,
                      target_dates_list=fig_target_date_list,
                      verbose=fig_verbose)
metric_dic = {task: metric_dic[(task,target_dates)] for task, target_dates in metric_dic.keys()}

if False:
    print_improvements(metric_dic, 
                       model_name='pbc_debias_aifs', 
                       baseline_models=['debiased_aifs'])

fig_show=True
fig_save=True
figure_source_data = True
figure_source_data_filename = "fig3_pbc_ai.xlsx"
figure_source_data_sheet_prefix="fig3a"

plot_rpss_barplot(metric_dic,
                   model_names=fig_model_names,
                   target_dates=fig_target_dates,
                   show_fig=fig_show,
                   source_data=figure_source_data,
                   source_data_filename=figure_source_data_filename,
                   source_data_sheet_prefix=figure_source_data_sheet_prefix,
                   save_fig=fig_save) 

#### Fig. 3b: Raw ECMWF, PoET, PBC-PoET barplots by task (2024)

In [ ]:
fig_model_names=['climatology', 'ecmwf', 'debiased_ecmwf', 'msn', 'pbc_msn']
fig_model_names_str="MSN models"
fig_gt_ids=["era5-tas", "era5-pr", "era5-mslp"]
fig_horizons=["19", "26"]
fig_target_dates=["std_msn_forecast"]
fig_verbose=False

metric_dic = get_all_rps(model_names=fig_model_names,
                      model_names_str=fig_model_names_str,
                      gt_ids=fig_gt_ids,
                      horizons=fig_horizons,
                      target_dates_list=fig_target_dates,
                      verbose=fig_verbose)
metric_dic = {task: metric_dic[(task,target_dates)] for task, target_dates in metric_dic.keys()}


if False:
    print_improvements(metric_dic, 
                       model_name='pbc_msn', 
                       baseline_models=['msn', 'debiased_ecmwf'])

fig_model_names=['climatology', 'ecmwf', 'debiased_ecmwf', 'msn', 'pbc_msn']
fig_target_dates="std_msn_forecast"
fig_show=True
fig_save=True
fig_verbose=False
figure_source_data = True
figure_source_data_filename = "fig3_pbc_ai.xlsx"
figure_source_data_sheet_prefix="fig3b"

plot_rpss_barplot(metric_dic,
                   model_names=fig_model_names,
                   target_dates=fig_target_dates,
                   show_fig=fig_show,
                   save_fig=fig_save,
                   source_data=figure_source_data,
                   source_data_filename=figure_source_data_filename,
                   source_data_sheet_prefix=figure_source_data_sheet_prefix,
                   verbose=fig_verbose)

## Fig. 5: Forecasting extreme weather with the leading dynamical model (ECMWF) and its probabilistic bias correction (PBC). 

#### Fig. 5a: Raw ECMWF, Debiased ECMWF, PBC-ECMWF extreme barplots (2016-2024)

In [ ]:
fig_model_names=['climatology', 'ecmwf', 'debiased_ecmwf', 'pbc_ecmwf_combo']
fig_model_names_str="ECMWF models"
fig_target_dates="std_test"
fig_target_dates_list=[fig_target_dates]
fig_verbose=False
figure_source_data = True
figure_source_data_filename = "fig5_extremes3.xlsx"
figure_source_data_sheet_prefix = "fig5a"

fig_prefixes = ["F95_","F5_"]
fig_horizons=["19"]
fig_gt_ids = []
for prefix in fig_prefixes:
    for var in ["tas", "pr", "mslp"]:
        fig_gt_ids.append(f"era5-{prefix}{var}")

tic()
all_wtd_mse = get_all_metrics(
    model_names=fig_model_names,
    model_names_str=fig_model_names_str,
    metrics=['wtd_mse'],
    gt_ids=fig_gt_ids,
    horizons=fig_horizons,
    target_dates_list=fig_target_dates_list,
    common_dates=True,
    verbose=fig_verbose)

# Index all_wtd_mse by task alone
all_wtd_mse = {t: all_wtd_mse[(m, t, d)] for m, t, d in all_wtd_mse.keys()}
toc()
for horizon in fig_horizons:
    fig_show=True
    fig_save=True
    plot_single_horizon_bss_barplot(all_wtd_mse,
                    horizon = horizon,
                    model_names=fig_model_names,
                    target_dates=fig_target_dates,
                    show_fig=fig_show,
                    save_fig=fig_save,
                    prefixes=fig_prefixes,
                    verbose=fig_verbose,
                    source_data=figure_source_data,
                    source_data_filename=figure_source_data_filename,
                    source_data_sheet_prefix=figure_source_data_sheet_prefix,
                    y_bottom=-0.113 if horizon=="19" else -0.145,)

if False:
    print_improvements(all_wtd_mse, 
                        model_name='pbc_ecmwf_combo', 
                        baseline_models=['ecmwf', 'debiased_ecmwf'])

#### Fig. 5b: Raw ECMWF, Debiased ECMWF, PBC-ECMWF extreme map plots (2016-2024)

In [ ]:
fig_model_names=['ecmwf', 'debiased_ecmwf', 'pbc_ecmwf_combo']
fig_target_dates="std_test"
diff_cmap = "bwr"
skill_cmap = "RdBu_r"
figure_show = True
figure_save = True
prefixes = ["F95_","F5_"]
figure_source_data = True
figure_source_data_filename = "fig5_extremes3.xlsx"
figure_source_data_sheet_prefix = "fig5c"

for horizon in [19]:
    plot_single_horizon_bss_diff_grid_6x4(
        horizon = horizon, 
        prefixes = prefixes,
        model_names=fig_model_names, 
        target_dates=fig_target_dates,
        diff_cmap=diff_cmap, 
        skill_cmap=skill_cmap, 
        show_fig=figure_show, 
        source_data=figure_source_data,
        source_data_filename=figure_source_data_filename,
        source_data_sheet_prefix=figure_source_data_sheet_prefix,
        save_fig=figure_save)

#### Fig. 5c: Raw ECMWF, Debiased ECMWF, PBC-ECMWF flood forecasting (2016-2026)

In [ ]:
figure_events_json=Path("eval/viz/bss-barplot-floods/data/flood-events-gdacs-2016_2026.json")
figure_percentile=95
figure_use_all_dates=True
figure_recompute=False
figure_save_fig=True
figure_show_fig=True
figure_source_data = True
figure_source_data_filename = "fig5_extremes3.xlsx"
figure_source_data_sheet_prefix = "fig5b"


fig, bss, brier_data = generate_flood_bss_barplot(
                                        events_json=figure_events_json,
                                        percentile=figure_percentile,
                                        use_all_dates=figure_use_all_dates,
                                        recompute=figure_recompute,
                                        save_fig=figure_save_fig,
                                        show_fig=figure_show_fig,
                                        source_data=figure_source_data,
                                        source_data_filename=figure_source_data_filename,
                                        source_data_sheet_prefix=figure_source_data_sheet_prefix,
                                )

# Extended Data

## Extended Data Figure 1: Global forecast skill per season for the leading dynamical model (ECMWF) and its probabilistic bias correction (PBC). 

#### Extended Data Figure 1: Raw ECMWF, Debiased ECMWF, PBC-ECMWF RPSS by season barplots (2016-2024)

In [ ]:
fig_model_names=["climatology", "ecmwf", "debiased_ecmwf", "pbc_ecmwf_combo"]
fig_model_names_str="ECMWF-based models"
fig_gt_ids=["era5-tas", "era5-pr", "era5-mslp"]
fig_horizons=["19", "26"]
fig_target_dates=["std_test"]
fig_verbose=False

metrics_dic = get_all_rps(model_names=fig_model_names,
                      model_names_str=fig_model_names_str,
                      gt_ids=fig_gt_ids,
                      horizons=fig_horizons,
                      target_dates_list=fig_target_dates,
                      verbose=fig_verbose)
metrics_dic = {task: metrics_dic[(task,target_dates)] for task, target_dates in metrics_dic.keys()}

# Set figure parameters
figure_model_names = ["ecmwf", "debiased_ecmwf", "pbc_ecmwf_combo"]
figure_gt_ids = ['era5-tas', 'era5-pr', 'era5-mslp']
figure_horizons = [19, 26]
figure_target_dates = 'std_test'
figure_show = True
figure_save = True
figure_verbose = False
figure_source_data = True
figure_source_data_filename = "ed_fig1_rpss_by_season_std_test.xlsx"


plot_seasonal_rpss_grouped_bar(metrics_dic,
                                model_names=figure_model_names,
                                gt_ids=figure_gt_ids,
                                horizons=figure_horizons,
                                target_dates=figure_target_dates,
                                show_fig=figure_show,
                                save_fig=figure_save,
                                source_data=figure_source_data,
                                source_data_filename=figure_source_data_filename,
                                verbose=figure_verbose)

## Extended Data Figure 2: Regional forecast skill of the leading dynamical model (ECMWF) and its probabilistic bias correction (PBC). 

#### Extended Data Figure 2: Raw ECMWF, Deb. ECMWF, PBC-ECMWF RPSS barplots by region (std_test: 2016-2024)

In [ ]:
fig_model_names=['climatology', 'ecmwf', 'debiased_ecmwf', 'pbc_ecmwf_combo']###
fig_gt_ids = ['era5-tas', 'era5-pr', 'era5-mslp']
fig_horizons = [19, 26]
fig_target_dates = ["std_test"]
fig_regions = 'all' 
fig_verbose=False

metrics_dic = get_all_rps(
    model_names = fig_model_names,
    model_names_str="ECMWF-based models",
    horizons = fig_horizons,
    target_dates_list = fig_target_dates,
    regions = fig_regions,
    verbose=fig_verbose)
metrics_dic = {task: metrics_dic[(task,target_dates)] for task, target_dates in metrics_dic.keys()}


fig_model_names = ['ecmwf', 'debiased_ecmwf', 'pbc_ecmwf_combo']
fig_gt_ids = ['era5-tas', 'era5-pr', 'era5-mslp']
fig_horizons = [19, 26]
fig_target_dates = "std_test"
fig_regions = 'all' 
fig_show = True
fig_save = True
fig_verbose = False
figure_source_data = True
figure_source_data_filename = "ed_fig2_regional_rpss_std_test.xlsx"

plot_rpss_by_region_all(metrics_dic,
                       model_names=fig_model_names,
                       gt_ids=fig_gt_ids,
                       horizons=fig_horizons,
                       target_dates=fig_target_dates,
                       regions=fig_regions,
                       show_fig=fig_show,
                       save_fig=fig_save,
                       source_data=figure_source_data,
                       source_data_filename=figure_source_data_filename,
                       verbose=fig_verbose)

## Extended Data Figure 3: Spatial distribution of precipitation model bias for the leading dynamical model (ECMWF) and its probabilistic bias correction (PBC). 

#### Extended Data Figure 3: Raw ECMWF, Debiased ECMWF, PBC-ECMWF spatial bias (prediction-truth) (2016-2024)

In [ ]:
fig_model_names=["ecmwf", "debiased_ecmwf", "pbc_ecmwf_combo", "gt"]
fig_gt_ids = ["era5-pr"]
fig_horizons = [19, 26]
fig_target_dates="std_test"
fig_vmin=-0.2
fig_vmax=0.2
fig_show_fig=True
fig_save_fig=True
fig_verbose=False
figure_source_data = True
figure_source_data_filename = "ed_fig3_bias_pr.xlsx"

results_dict = get_all_preds(model_names=fig_model_names,
                  gt_ids=fig_gt_ids,
                  horizons=fig_horizons,
                  fs=[1, 2, 3, 4],
                  target_dates="std_test",
                  verbose=fig_verbose)

for fig_gt_id, fig_horizon in product(fig_gt_ids, fig_horizons):
    fig_show_cbar = (fig_horizon==26)
    figure_source_data_sheet_prefix=f"Fig_s4_{fig_gt_id}_{fig_horizon}"
    plot_bias_maps_3x4(results_dict,  # Now taking the dictionary
                       model_names=fig_model_names,
                          gt_id=fig_gt_id,
                          horizon=fig_horizon,
                          fs=[1, 2, 3, 4],
                          cmap="RdBu_r",
                          vmin = fig_vmin,
                        vmax = fig_vmax,
                        show_cbar=fig_show_cbar,
                        show_fig = fig_show_fig,
                        source_data=figure_source_data,
                        source_data_filename=figure_source_data_filename,
                        source_data_sheet_prefix=figure_source_data_sheet_prefix,
                        save_fig = fig_save_fig
                    )

In [ ]:
if False:
    fig_model_names=["ecmwf", "debiased_ecmwf", "pbc_ecmwf_combo"]
    fig_gt_ids = ["era5-pr"]
    fig_horizons = [19, 26]
    figure_source_data = True
    figure_source_data_filename = "ed_fig3_bias_pr.xlsx"

    for fig_gt_id, fig_horizon in product(fig_gt_ids, fig_horizons):
        figure_source_data_sheet_prefix=f"Fig_s4_{fig_gt_id}_{fig_horizon}"
        results = print_model_bias(results_dict, 
                        model_names=fig_model_names,
                        gt_id=fig_gt_id,
                        horizon=fig_horizon,
                        fs=[1, 2, 3, 4],
                        source_data=figure_source_data,
                        source_data_filename=figure_source_data_filename,
                        source_data_sheet_prefix=figure_source_data_sheet_prefix)

## Extended Data Figure 4: Spatial distribution of temperature model bias for the leading dynamical model (ECMWF) and its probabilistic bias correction (PBC).

#### Extended Data Figure 4: Raw ECMWF, Debiased ECMWF, PBC-ECMWF spatial bias (prediction-truth) (2016-2024) 

In [ ]:
fig_model_names=["ecmwf", "debiased_ecmwf", "pbc_ecmwf_combo", "gt"]
fig_gt_ids = ["era5-tas"]
fig_horizons = [19, 26]
fig_target_dates="std_test"
fig_vmin=-0.2
fig_vmax=0.2
fig_show_fig=True
fig_save_fig=True
fig_verbose=False
figure_source_data = True
figure_source_data_filename = "ed_fig4_bias_tas.xlsx"

results_dict = get_all_preds(model_names=fig_model_names,
                  gt_ids=fig_gt_ids,
                  horizons=fig_horizons,
                  fs=[1, 2, 3, 4],
                  target_dates="std_test",
                  verbose=fig_verbose)

for fig_gt_id, fig_horizon in product(fig_gt_ids, fig_horizons):
    fig_show_cbar = (fig_horizon==26)
    figure_source_data_sheet_prefix=f"Fig_s4_{fig_gt_id}_{fig_horizon}"
    plot_bias_maps_3x4(results_dict,  # Now taking the dictionary
                       model_names=fig_model_names,
                          gt_id=fig_gt_id,
                          horizon=fig_horizon,
                          fs=[1, 2, 3, 4],
                          cmap="RdBu_r",
                          vmin = fig_vmin,
                        vmax = fig_vmax,
                        show_cbar=fig_show_cbar,
                        show_fig = fig_show_fig,
                        source_data=figure_source_data,
                        source_data_filename=figure_source_data_filename,
                        source_data_sheet_prefix=figure_source_data_sheet_prefix,
                        save_fig = fig_save_fig
                    )

In [ ]:
if False:
    fig_model_names=["ecmwf", "debiased_ecmwf", "pbc_ecmwf_combo"]
    fig_gt_ids = ["era5-tas"]
    fig_horizons = [19, 26]
    figure_source_data = True
    figure_source_data_filename = "ed_fig4_bias_tas.xlsx"

    for fig_gt_id, fig_horizon in product(fig_gt_ids, fig_horizons):
        figure_source_data_sheet_prefix=f"Fig_s4_{fig_gt_id}_{fig_horizon}"
        results = print_model_bias(results_dict, 
                        model_names=fig_model_names,
                        gt_id=fig_gt_id,
                        horizon=fig_horizon,
                        fs=[1, 2, 3, 4],
                        source_data=figure_source_data,
                        source_data_filename=figure_source_data_filename,
                        source_data_sheet_prefix=figure_source_data_sheet_prefix)

## Extended Data Figure 5: Spatial distribution of mean sea level pressure model bias for the leading dynamical model (ECMWF) and its probabilistic bias correction (PBC).

#### Extended Data Figure 5: Raw ECMWF, Debiased ECMWF, PBC-ECMWF spatial bias (prediction-truth) (2016-2024) 

In [ ]:
fig_model_names=["ecmwf", "debiased_ecmwf", "pbc_ecmwf_combo", "gt"]
fig_gt_ids = ["era5-mslp"]
fig_horizons = [19, 26]
fig_target_dates="std_test"
fig_vmin=-0.2
fig_vmax=0.2
fig_show_fig=True
fig_save_fig=True
fig_verbose=False
figure_source_data = True
figure_source_data_filename = "ed_fig5_bias_mslp.xlsx"

results_dict = get_all_preds(model_names=fig_model_names,
                  gt_ids=fig_gt_ids,
                  horizons=fig_horizons,
                  fs=[1, 2, 3, 4],
                  target_dates="std_test",
                  verbose=fig_verbose)

for fig_gt_id, fig_horizon in product(fig_gt_ids, fig_horizons):
    fig_show_cbar = (fig_horizon==26)
    figure_source_data_sheet_prefix=f"Fig_s4_{fig_gt_id}_{fig_horizon}"
    plot_bias_maps_3x4(results_dict,  # Now taking the dictionary
                       model_names=fig_model_names,
                          gt_id=fig_gt_id,
                          horizon=fig_horizon,
                          fs=[1, 2, 3, 4],
                          cmap="RdBu_r",
                          vmin = fig_vmin,
                        vmax = fig_vmax,
                        show_cbar=fig_show_cbar,
                        show_fig = fig_show_fig,
                        source_data=figure_source_data,
                        source_data_filename=figure_source_data_filename,
                        source_data_sheet_prefix=figure_source_data_sheet_prefix,
                        save_fig = fig_save_fig
                    )

In [ ]:
if False:
    fig_model_names=["ecmwf", "debiased_ecmwf", "pbc_ecmwf_combo"]
    fig_gt_ids = ["era5-mslp"]
    fig_horizons = [19, 26]
    figure_source_data = True
    figure_source_data_filename = "ed_fig5_bias_mslp.xlsx"

    for fig_gt_id, fig_horizon in product(fig_gt_ids, fig_horizons):
        figure_source_data_sheet_prefix=f"Fig_s4_{fig_gt_id}_{fig_horizon}"
        results = print_model_bias(results_dict, 
                        model_names=fig_model_names,
                        gt_id=fig_gt_id,
                        horizon=fig_horizon,
                        fs=[1, 2, 3, 4],
                        source_data=figure_source_data,
                        source_data_filename=figure_source_data_filename,
                        source_data_sheet_prefix=figure_source_data_sheet_prefix,)

## Extended Data Figure 6: Global forecast skill of FuXi-S2S and PBC-ECMWF. 

#### Extended Data Figure 6a: Fuxi, PBC-ECMWF barplots (2017-2021)

In [ ]:
fig_model_names=['climatology', 'debiased_fuxi', 'pbc_ecmwf_combo']
fig_model_names_str="Fuxi models"
fig_gt_ids=["era5-tas", "era5-pr", "era5-mslp"]
fig_horizons=["19", "26"]
fig_target_dates=["std_fuxi"]
common_dates=False
fig_verbose=False

metric_dic = get_all_rps(model_names=fig_model_names,
                      model_names_str=fig_model_names_str,
                      gt_ids=fig_gt_ids,
                      horizons=fig_horizons,
                      target_dates_list=fig_target_dates,
                      common_dates=common_dates,
                      verbose=fig_verbose)
metric_dic = {task: metric_dic[(task,target_dates)] for task, target_dates in metric_dic.keys()}

if False:
    print_improvements(metric_dic, 
                       model_name='pbc_ecmwf_combo', 
                       baseline_models=['debiased_fuxi'])

fig_model_names=['climatology', 'debiased_fuxi', 'pbc_ecmwf_combo']
fig_target_dates="std_fuxi"
fig_show=True
fig_save=True
fig_verbose=False
figure_source_data = True
figure_source_data_filename = "ed_fig6_fuxi.xlsx"
plot_rpss_ci_barplot(metric_dic,
                     model_names=fig_model_names,
                     target_dates=fig_target_dates,
                     show_fig=fig_show,
                     save_fig=fig_save,
                     source_data=figure_source_data,
                     source_data_filename=figure_source_data_filename,
                     verbose=fig_verbose)

#### Extended Data Figure 6b: FuXi, PBC-ECMWF diff maps (2017-2021)

In [ ]:
# Set figure parameters
figure_model_names = ['debiased_fuxi', 'pbc_ecmwf_combo']
figure_gt_ids = ['era5-tas', 'era5-pr', 'era5-mslp']
figure_horizons = [19, 26]
figure_metric = 'lat_lon_rpss'
figure_target_dates = 'std_fuxi'
diff_cmap = "bwr"
skill_cmap = "RdBu_r"
print_mean=False
figure_show = True
figure_save = True
figure_source_data = True
figure_source_data_filename = "ed_fig6_fuxi.xlsx"

plot_metric_diff_grid_6x3(model_names=figure_model_names,
                          gt_ids=figure_gt_ids,
                          horizons=figure_horizons,
                          metric=figure_metric,
                          target_dates=figure_target_dates,
                          diff_cmap=diff_cmap,
                          skill_cmap=skill_cmap, 
                          show_fig=figure_show,
                          source_data=figure_source_data,
                          source_data_filename=figure_source_data_filename,
                          save_fig=figure_save)

## Extended Data Figure 7: Global forecast skill of the leading dynamical model (ECMWF), the hybrid (AI + dynamical) model PoET, and the components of their probabilistic bias corrections (PBC). 

#### Extended Data Figure 7a: Raw ECMWF, Debiased ECMWF, Persistence++-ECMWF, Debias++-ECMWF, PBC-ECMWF barplots by task (2024)

In [ ]:
fig_model_names=['climatology', 'ecmwf', 'debiased_ecmwf',
                 'tuned_ecmwfpp', 'proj_tuned_ecmwfpp',
                 'perpp_ecmwf', 'proj_perpp_ecmwf',
                 'perpp_debias', 'proj_perpp_debias',
                 'pbc_ecmwf_combo']

fig_model_names_str="PBC-ECMWF model components"
fig_gt_ids=["era5-tas", "era5-pr", "era5-mslp"]
fig_horizons=["19", "26"]
fig_target_dates="std_test"
fig_target_dates_list=[fig_target_dates]
fig_verbose=False
figure_source_data = True
figure_source_data_filename = "ed_fig7_pbc_breakdown.xlsx"
figure_source_data_sheet_prefix="ed_fig7a"

metric_dic = get_all_rps(model_names=fig_model_names,
                      model_names_str=fig_model_names_str,
                      gt_ids=fig_gt_ids,
                      horizons=fig_horizons,
                      target_dates_list=fig_target_dates_list,
                      verbose=fig_verbose)
metric_dic = {task: metric_dic[(task,target_dates)] for task, target_dates in metric_dic.keys()}

fig_variable_models = {
        "Temperature": [
            "ecmwf",
            "debiased_ecmwf",
            "tuned_ecmwfpp",
            "proj_tuned_ecmwfpp",
            "perpp_debias",
            "proj_perpp_debias",
            "pbc_ecmwf_combo",
        ],
        "Precipitation": [
            "ecmwf",
            "debiased_ecmwf",
            "tuned_ecmwfpp",
            "proj_tuned_ecmwfpp",
            "perpp_ecmwf",
            "proj_perpp_ecmwf",
            "pbc_ecmwf_combo",
        ],
        "Sea Level Pressure": [
            "ecmwf",
            "debiased_ecmwf",
            "tuned_ecmwfpp",
            "proj_tuned_ecmwfpp",
            "perpp_debias",
            "proj_perpp_debias",
            "pbc_ecmwf_combo",
        ],
    }
fig_show=True
fig_save=True
fig_by_season=False
fig_verbose=False
fig_show=True
fig_save=True
fig_by_season=False
fig_verbose=False
fig_legend_order=["ecmwf",
            "debiased_ecmwf",
            "tuned_ecmwfpp",
            "proj_tuned_ecmwfpp",
            "perpp_ecmwf",
            "proj_perpp_ecmwf",
            "pbc_ecmwf_combo",
                 ]
plot_rpss_barplot(metric_dic,
                   model_names=fig_model_names,
                   baseline_models=['ecmwf', 'debiased_ecmwf'],
                   variable_models=fig_variable_models,
                   target_dates=fig_target_dates,
                   show_fig=fig_show,
                   save_fig=fig_save,
                   by_season=fig_by_season,
                   legend_order=fig_legend_order,
                   legend_location="top",
                   legend_ncols=3,
                   source_data=figure_source_data,
                   source_data_filename=figure_source_data_filename,
                   source_data_sheet_prefix=figure_source_data_sheet_prefix,
                   verbose=fig_verbose,
                   suffix='_breakdown')


#### Extended Data Figure 7b: Raw ECMWF, Debiased ECMWF, PoET, Persistence++-PoET, Debias++-PoET, PBC-PoET barplots by task (2024)

In [ ]:
fig_model_names=['climatology', 'ecmwf', 'debiased_ecmwf', 'msn', 'proj_perpp_msn', 'proj_tuned_msnpp', 'pbc_msn']
fig_model_names_str="PBC-PoET model components"
fig_gt_ids=["era5-tas", "era5-pr", "era5-mslp"]
fig_horizons=["19", "26"]
fig_target_dates="std_msn_forecast"
fig_target_dates_list=[fig_target_dates]
fig_verbose=False
figure_source_data = True
figure_source_data_filename = "ed_fig7_pbc_breakdown.xlsx"
figure_source_data_sheet_prefix="ed_fig7b"

metric_dic = get_all_rps(model_names=fig_model_names,
                      model_names_str=fig_model_names_str,
                      gt_ids=fig_gt_ids,
                      horizons=fig_horizons,
                      target_dates_list=fig_target_dates_list,
                      verbose=fig_verbose)
metric_dic = {task: metric_dic[(task,target_dates)] for task, target_dates in metric_dic.keys()}

fig_show=True
fig_save=True
fig_by_season=False
fig_verbose=False
plot_rpss_barplot(metric_dic,
                model_names=fig_model_names,
                baseline_models=['ecmwf', 'debiased_ecmwf', 'msn'],
                target_dates=fig_target_dates,
                show_fig=fig_show,
                save_fig=fig_save,
                by_season=fig_by_season,
                source_data=figure_source_data,
                source_data_filename=figure_source_data_filename,
                source_data_sheet_prefix=figure_source_data_sheet_prefix,
                verbose=fig_verbose,
                suffix='_breakdown')

## Extended Data Figure 8: Forecasting the December 2025 cold air outbreak in the Eastern United States.

#### Extended Data Figure 8: December 2025 cold air outbreak in the Eastern United States

In [ ]:
if True:
    fig_gt_id = 'era5-tas'
    fig_horizon = 26
    fig_target_date = None
    fig_issuance_date = '20251120'
    fig_team_name = 'Dynamical_S2SDatabase'
    fig_model_name = 'ECMWF'
    fig_bbox_name = 'us'
    fig_quintile = 0.2
    fig_y_suptitle = 0.89
    figure_source_data = True
    figure_source_data_filename = "ed_fig8_extreme_weather_tas_20251120_p2_us.xlsx"
    
    
    plot_probability_maps(
        gt_id = fig_gt_id,
        horizon = fig_horizon,
        target_date = fig_target_date,
        issuance_date = fig_issuance_date,
        team_name = fig_team_name,
        model_name = fig_model_name,
        bbox_name = fig_bbox_name, 
        quintile = fig_quintile,
        y_suptitle = fig_y_suptitle,
        source_data=figure_source_data,
        source_data_filename=figure_source_data_filename,
    )

## Extended Data Figure 9: Forecasting extreme weather with the leading dynamical model (ECMWF) and its probabilistic bias correction (PBC) in week 4. 

#### Extended Data Figure 9a: Raw ECMWF, Debiased ECMWF, PBC-ECMWF extreme barplots (2016-2024)

In [ ]:
fig_model_names=['climatology', 'ecmwf', 'debiased_ecmwf', 'pbc_ecmwf_combo']
fig_model_names_str="ECMWF models"
fig_target_dates="std_test"
fig_target_dates_list=[fig_target_dates]
fig_verbose=False
figure_source_data = True
figure_source_data_filename = "ed_fig9_extremes4.xlsx"
figure_source_data_sheet_prefix="ed_fig9a"
fig_prefixes = ["F95_","F5_"]
fig_horizons=["26"]
fig_gt_ids = []
for prefix in fig_prefixes:
    for var in ["tas", "pr", "mslp"]:
        fig_gt_ids.append(f"era5-{prefix}{var}")

tic()
all_wtd_mse = get_all_metrics(
    model_names=fig_model_names,
    model_names_str=fig_model_names_str,
    metrics=['wtd_mse'],
    gt_ids=fig_gt_ids,
    horizons=fig_horizons,
    target_dates_list=fig_target_dates_list,
    common_dates=True,
    verbose=fig_verbose)
# Index all_wtd_mse by task alone
all_wtd_mse = {t: all_wtd_mse[(m, t, d)] for m, t, d in all_wtd_mse.keys()}
toc()
for horizon in fig_horizons:
    fig_show=True
    fig_save=True
    plot_single_horizon_bss_barplot(all_wtd_mse,
                    horizon = horizon,
                    model_names=fig_model_names,
                    target_dates=fig_target_dates,
                    show_fig=fig_show,
                    save_fig=fig_save,
                    prefixes=fig_prefixes,
                    source_data=figure_source_data,
                    source_data_filename=figure_source_data_filename,
                    source_data_sheet_prefix=figure_source_data_sheet_prefix,
                    verbose=fig_verbose,
                    y_bottom=-0.113 if horizon=="19" else -0.145,)

if False:
    print_improvements(all_wtd_mse, 
                        model_name='pbc_ecmwf_combo', 
                        baseline_models=['ecmwf', 'debiased_ecmwf'])


#### Extended Data Figure 9b: Raw ECMWF, Debiased ECMWF, PBC-ECMWF extreme map plots (2016-2024)

In [ ]:
fig_model_names=['ecmwf', 'debiased_ecmwf', 'pbc_ecmwf_combo']
fig_target_dates="std_test"
diff_cmap = "bwr"
skill_cmap = "RdBu_r"
figure_show = True
figure_save = True
figure_source_data = True
figure_source_data_filename = "ed_fig9_extremes4.xlsx"
figure_source_data_sheet_prefix="ed_fig9b"
prefixes = ["F95_","F5_"]

for horizon in [26]:
    plot_single_horizon_bss_diff_grid_6x4(
        horizon = horizon, 
        prefixes = prefixes,
        model_names=fig_model_names, 
        target_dates=fig_target_dates,
        diff_cmap=diff_cmap, 
        skill_cmap=skill_cmap, 
        show_fig=figure_show, 
        source_data=figure_source_data,
        source_data_filename=figure_source_data_filename,
        source_data_sheet_prefix=figure_source_data_sheet_prefix,
        save_fig=figure_save)